# Sequence Labeling and BIO Tagging

In this notebook, we explore why Named Entity Recognition (NER) is not simply a token classification problem.

We introduce:

- BIO tagging
- sequence labeling
- structured prediction
- transition constraints
- the intuition behind CRFs

## Learning Objectives

By the end of this notebook, you should be able to:
- explain BIO tagging
- understand why independent token classification fails
- identify invalid label sequences
- explain why NER is a structured prediction problem
- understand the intuition behind CRFs

---

## 1. Why Token Classification is Not Enough

Suppose we want to detect entities in the sentence:

```text
Barack Obama visited New York City.
```

A simple classifier could predict a label for each token independently.

But entity labels are not independent.

Example:

```text
Barack → PERSON
Obama  → O
```

This is inconsistent because `Obama` is clearly part of the same entity.

NER therefore requires sequence-aware prediction.

## 2. BIO Tagging Scheme

BIO tagging is a common way to represent entity spans.

| Prefix | Meaning |
|---|---|
| B | Beginning of entity |
| I | Inside entity |
| O | Outside entity |

Example:

| Token | Label |
|---|---|
| Barack | B-PER |
| Obama | I-PER |
| visited | O |
| New | B-GPE |
| York | I-GPE |
| City | I-GPE |
| . | O |


In [1]:
import pandas as pd

tokens = ["Barack", "Obama", "visited", "New", "York", "City", "."]
labels = ["B-PER", "I-PER", "O", "B-GPE", "I-GPE", "I-GPE", "O"]

pd.DataFrame({
    "token": tokens,
    "BIO_label": labels
})

,token,BIO_label
0,Barack,B-PER
1,Obama,I-PER
2,visited,O
3,New,B-GPE
4,York,I-GPE
5,City,I-GPE
6,.,O


## 3. Multi-Token Entities

Many named entities contain multiple tokens.

Examples:

- New York City
- San Francisco
- Barack Obama
- United Nations

A model must therefore learn:

- entity boundaries
- entity continuation
- label consistency


In [2]:
examples = [
    (["New", "York", "City"], ["B-GPE", "I-GPE", "I-GPE"]),
    (["Barack", "Obama"], ["B-PER", "I-PER"]),
    (["United", "Nations"], ["B-ORG", "I-ORG"])
]

for tokens, labels in examples:
    print("Tokens:", tokens)
    print("Labels:", labels)
    print("-" * 60)

Tokens: ['New', 'York', 'City']
Labels: ['B-GPE', 'I-GPE', 'I-GPE']
------------------------------------------------------------
Tokens: ['Barack', 'Obama']
Labels: ['B-PER', 'I-PER']
------------------------------------------------------------
Tokens: ['United', 'Nations']
Labels: ['B-ORG', 'I-ORG']
------------------------------------------------------------


## 4. Independent Token Prediction Failure

Suppose a classifier predicts labels independently.

The model might produce:

| Token | Predicted Label |
|---|---|
| Barack | B-PER |
| Obama | O |
| visited | O |
| New | B-GPE |
| York | O |
| City | I-GPE |

These predictions are inconsistent.

The model fails to preserve valid entity spans.

In [3]:
bad_predictions = pd.DataFrame({
    "token": ["Barack", "Obama", "visited", "New", "York", "City"],
    "predicted_label": ["B-PER", "O", "O", "B-GPE", "O", "I-GPE"]
})

bad_predictions

,token,predicted_label
0,Barack,B-PER
1,Obama,O
2,visited,O
3,New,B-GPE
4,York,O
5,City,I-GPE


## 5. BIO Validity Rules

BIO tagging imposes structural constraints.

Examples:

### Valid

```text
B-PER → I-PER
```

### Invalid

```text
B-PER → I-ORG
```

### Invalid

```text
O → I-PER
```

Sequence-aware models learn these constraints automatically.

In [4]:
def is_valid_transition(prev_label, current_label):
    """Simple BIO transition validation."""

    if current_label.startswith("I-"):
        entity_type = current_label[2:]

        if prev_label == "O":
            return False

        if prev_label[2:] != entity_type:
            return False

    return True

In [5]:
tests = [
    ("B-PER", "I-PER"),
    ("B-PER", "I-ORG"),
    ("O", "I-PER"),
    ("B-GPE", "I-GPE"),
]

for prev_label, current_label in tests:
    valid = is_valid_transition(prev_label, current_label)
    print(f"{prev_label:10s} -> {current_label:10s} | valid = {valid}")

B-PER      -> I-PER      | valid = True
B-PER      -> I-ORG      | valid = False
O          -> I-PER      | valid = False
B-GPE      -> I-GPE      | valid = True


## 6. Sequence Labeling

NER is therefore not only about predicting labels.

The model must predict:

- valid sequences
- consistent entity spans
- neighboring label dependencies

This is called:

# Structured Prediction

Instead of predicting labels independently, the model predicts the entire sequence jointly.

## 7. CRF Intuition

Conditional Random Fields (CRFs) became one of the most important classical NLP models for NER.

CRFs combine:

- handcrafted token features
- contextual information
- sequence consistency

Key idea:

```text
B-PER is likely followed by I-PER
```

The model learns transition probabilities between labels.

In [6]:
transition_examples = pd.DataFrame({
    "previous_label": ["B-PER", "B-GPE", "O", "B-ORG"],
    "next_label": ["I-PER", "I-GPE", "O", "I-ORG"],
    "likely": [True, True, True, True]
})

transition_examples

,previous_label,next_label,likely
0,B-PER,I-PER,True
1,B-GPE,I-GPE,True
2,O,O,True
3,B-ORG,I-ORG,True


## 8. Independent Prediction vs Structured Prediction

| Independent Token Classification | Structured Prediction |
|---|---|
| Each token predicted separately | Entire sequence predicted jointly |
| Can create invalid sequences | Learns sequence constraints |
| Ignores neighboring labels | Uses label dependencies |
| Simpler | More consistent |

CRFs improved classical NER systems significantly before deep learning became dominant.

## 9. Transition to Deep Learning

Classical sequence models still relied heavily on handcrafted features.

Deep learning later replaced much of this feature engineering with:

- learned embeddings
- recurrent neural networks
- transformers
- contextual representations

But the core NER problem remained the same:

> Predict consistent label sequences for text.

## 10. Mini Exercise

Consider the following sentence:

```text
Elon Musk visited San Francisco in 2023.
```

Tasks:

1. Create BIO labels manually.
2. Identify multi-token entities.
3. Create an example of an invalid BIO sequence.
4. Explain why the invalid sequence is problematic.


In [7]:
# TODO

tokens = ["Elon", "Musk", "visited", "San", "Francisco", "in", "2023", "."]

# Create your BIO labels here
labels = []

pd.DataFrame({
    "token": tokens,
    "BIO_label": labels
})

ValueError: All arrays must be of the same length

## 11. Reflection

Answer briefly:

1. Why is token classification alone insufficient for NER?
2. Why do multi-token entities create difficulties?
3. What kinds of invalid sequences can occur?
4. Why are neighboring labels important?
5. What problem do CRFs solve?
6. Why are sequence-aware models useful for IE tasks?


## Summary

In this notebook, we explored:

- BIO tagging
- multi-token entities
- sequence labeling
- structured prediction
- CRF intuition

Main takeaway:

> NER is not only about classifying tokens. It is about predicting consistent label sequences.

This idea became central for:

- CRFs
- RNN-based NER
- Transformer-based NER
- modern LLM-based IE systems
